In [12]:
import sys
import os

from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))

from src.cv import *

X, y, X_test, test_ids = load_data(data_dir="../data")
X = prepare_categoricals(X, flavour="xgb")

_ = quick_cv(lambda: XGBClassifier(), X, y)

  fold 1/2  AUC=0.95744  (4s)  best_iter=99
  fold 2/2  AUC=0.95785  (3s)  best_iter=99
model
  fold AUCs   : 0.95744  0.95785
  mean +/- std: 0.95765 +/- 0.00020
  OOF AUC     : 0.95764   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 7.2s
  best_iters  : [99, 99]


In [13]:
"""
Try combinations of features
"""

X_temp = X.copy()

features_to_try = [
    ("total_screen", X["daily_screen_time_hours"] + X["weekend_screen_time"]),
    ("d_plus_2s", X["daily_screen_time_hours"] + 2 * X["social_media_hours"]),
    ("avg_screen", (5 * X["daily_screen_time_hours"] + 2 * X["weekend_screen_time"]) / 7),

    ("social_media_ratio", X["social_media_hours"] / (X["daily_screen_time_hours"] + 0.01)),
    ("portion_of_day_on_screen", X["daily_screen_time_hours"] / (24 - X["sleep_hours"])),
    ("screen_to_work_ratio", X["daily_screen_time_hours"] / (X["work_study_hours"] + 0.01)),
    ("leisure_ratio", (X["daily_screen_time_hours"] - X["work_study_hours"] - X["gaming_hours"]) / (X["daily_screen_time_hours"] + 0.01)),
    ("screen_share_of_free", (X["daily_screen_time_hours"] / (24 - X["sleep_hours"] - X["work_study_hours"]))),

    ("mins_per_open", X["daily_screen_time_hours"] * 60 / X["app_opens_per_day"]),
    ("notif_per_screen_hour", X["notifications_per_day"] / (X["daily_screen_time_hours"] + 0.01))
]

for name, col in features_to_try:
    X_temp[name] = col
    print(f"----- {name} -------")
    quick_cv(lambda: XGBClassifier(), X_temp, y)
    print("----------------------")
    X_temp = X.copy()


----- total_screen -------
  fold 1/2  AUC=0.95692  (8s)  best_iter=99
  fold 2/2  AUC=0.95793  (4s)  best_iter=99
model
  fold AUCs   : 0.95692  0.95793
  mean +/- std: 0.95743 +/- 0.00051
  OOF AUC     : 0.95743   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 12.5s
  best_iters  : [99, 99]
----------------------
----- d_plus_2s -------
  fold 1/2  AUC=0.95674  (6s)  best_iter=99
  fold 2/2  AUC=0.95791  (5s)  best_iter=99
model
  fold AUCs   : 0.95674  0.95791
  mean +/- std: 0.95733 +/- 0.00058
  OOF AUC     : 0.95733   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 11.9s
  best_iters  : [99, 99]
----------------------
----- avg_screen -------
  fold 1/2  AUC=0.95752  (2s)  best_iter=99
  fold 2/2  AUC=0.95752  (5s)  best_iter=99
model
  fold AUCs   : 0.95752  0.95752
  mean +/- std: 0.95752 +/- 0.00000
  OOF AUC     : 0.95752   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 7.2s
  best_

In [14]:
"""
Latent weights
"""
DAILY = "daily_screen_time_hours"
SOCIAL = "social_media_hours"
GAMING = "gaming_hours"
WORK = "work_study_hours"
SLEEP = "sleep_hours"
NOTIF = "notifications_per_day"
OPENS = "app_opens_per_day"
WEEKEND = "weekend_screen_time"

LATENT_W = {
    DAILY: 1.00,
    SOCIAL: 2.33,
    WEEKEND: 0.96,
    GAMING: -0.81,
    WORK: -0.71,
    SLEEP: 0.21,
}
X_temp = X.copy()
X_temp["latent"] = sum(w * X[c] for c, w in LATENT_W.items())
_ = quick_cv(lambda: XGBClassifier(), X_temp, y)

  fold 1/2  AUC=0.95694  (5s)  best_iter=99
  fold 2/2  AUC=0.95765  (2s)  best_iter=96
model
  fold AUCs   : 0.95694  0.95765
  mean +/- std: 0.95729 +/- 0.00036
  OOF AUC     : 0.95729   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 7.8s
  best_iters  : [99, 96]
